In [ ]:
# Edit only attached Kaggle Input paths and batch sizes.
from pathlib import Path

LEGALIR_SOURCE_PATH = Path("/kaggle/input/datasets/mduy2911/legalir/train.json")
CORPUS_PATH = Path("/kaggle/input/datasets/mduy2911/legalir/selected-contexts")
DENSE_MODEL_PATH = Path("/kaggle/input/datasets/mduy2911/bge-m3-kaggle")
RERANKER_MODEL_PATH = Path("/kaggle/input/datasets/mduy2911/bge-reranker-v2-m3-kaggle")

CORPUS_BATCH_SIZE = 256  # May be reduced for OOM; semantics do not change.
QUERY_BATCH_SIZE = 64    # May be reduced for OOM; semantics do not change.
RERANKER_BATCH_SIZE = 128  # May only be reduced for OOM.

DENSE_MODEL_NAME = "BAAI/bge-m3"
DENSE_DECLARED_REVISION = "5617a9f61b028005a4858fdac845db406aefb181"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
RERANKER_DECLARED_REVISION = "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e"
EXPECTED_SOURCE_SHA256 = "c39cde9e74977e350f1456e7d487aafe67d2bcbaa4fa26fcabd557fe635635b7"
EXPECTED_SPLIT_COUNTS = {"train": 4_941, "dev": 1_036, "holdout": 1_023}
EXPECTED_DOCUMENTS = 8_532
EXPECTED_CHUNKS = 199_816
EXPECTED_DENSE_RECALL_AT_100 = 0.9819015444015444
EXPECTED_M2_PRECISION = 0.18416988416988417
EXPECTED_M2_RECALL = 0.8647039897039897
EXPECTED_M2_MRR = 0.7291748808854571

CHUNK_SIZE = 2_000
CHUNK_OVERLAP = 200
CHUNK_STEP = 1_800
TOP_K_CHUNKS = 2_000
DOCUMENT_AGGREGATION = "sum_top_2_dense_chunk_scores"
CANDIDATE_DEPTH = 100
SUPPORT_POOL_SIZES = (2, 4, 8)
CE_SELECTED_CHUNKS = 2
DENSE_MAX_LENGTH = 8_192
RERANKER_MAX_SEQUENCE_LENGTH = 8_192
FINAL_K = 5
BOOTSTRAP_SEED = 20_260_913
BOOTSTRAP_RESAMPLES = 10_000
SANITY_ABS_TOLERANCE = 1e-9
RESULT_PATH = Path(
    "/kaggle/working/dense_cross_encoder_supporting_evidence_dev_results.json"
)


In [ ]:
# Enforce Internet-OFF execution and fail loudly for missing attached artifacts.
import os

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

for path, description, must_be_directory in (
    (LEGALIR_SOURCE_PATH, "LegalIR source JSON", False),
    (CORPUS_PATH, "LegalIR corpus directory", True),
    (DENSE_MODEL_PATH, "complete local BGE-M3 snapshot", True),
    (RERANKER_MODEL_PATH, "complete local BGE reranker snapshot", True),
):
    exists = path.is_dir() if must_be_directory else path.is_file()
    if not exists:
        raise FileNotFoundError(f"Attach the {description} at: {path}")


In [ ]:
# Standalone DEV-only implementation for within-document evidence selection.
import gc
import json
from collections import Counter, defaultdict
from hashlib import sha256
from math import isfinite
from statistics import median
from time import perf_counter

import numpy as np
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer


def read_json(path: Path):
    try:
        with path.open(encoding="utf-8-sig") as stream:
            return json.load(stream)
    except json.JSONDecodeError as exc:
        raise ValueError(f"{path}: invalid JSON: {exc}") from exc


def load_fixed_dev(path: Path) -> tuple[dict, dict]:
    source_hash = sha256(path.read_bytes()).hexdigest()
    if source_hash != EXPECTED_SOURCE_SHA256:
        raise RuntimeError(
            f"LegalIR source SHA-256 mismatch: expected {EXPECTED_SOURCE_SHA256}, "
            f"got {source_hash}. Stop before reading samples."
        )
    value = read_json(path)
    if not isinstance(value, dict) or not all(isinstance(v, dict) for v in value.values()):
        raise ValueError(f"{path}: expected an object keyed by sample ID")
    samples = {str(sample_id): sample for sample_id, sample in value.items()}
    if len(samples) != len(value):
        raise ValueError("duplicate sample IDs after string canonicalization")

    counts = {"train": 0, "dev": 0, "holdout": 0}
    dev_ids = []
    for sample_id, sample in samples.items():
        question = sample.get("question")
        group_key = question if isinstance(question, str) else f"\0fallback-sample-id:{sample_id}"
        bucket = int(sha256(group_key.encode("utf-8")).hexdigest()[:8], 16) % 100
        split_name = "train" if bucket < 70 else "dev" if bucket < 85 else "holdout"
        counts[split_name] += 1
        if split_name == "dev":
            dev_ids.append(sample_id)
    if counts != EXPECTED_SPLIT_COUNTS:
        raise RuntimeError(f"fixed split counts mismatch: {counts}. Stop; do not evaluate any split.")

    dev_ids.sort()
    dev = {sample_id: samples[sample_id] for sample_id in dev_ids}
    del samples, value
    for sample_id, sample in dev.items():
        if not isinstance(sample.get("question"), str):
            raise TypeError(f"DEV sample {sample_id!r}: question must be a string")
        if not isinstance(sample.get("answer"), list) or not sample["answer"]:
            raise ValueError(f"DEV sample {sample_id!r}: expected a non-empty answer list")
    if len(dev) != EXPECTED_SPLIT_COUNTS["dev"]:
        raise RuntimeError("DEV query-count mismatch")
    return dev, {
        "name": "fixed DEV",
        "queries": len(dev),
        "source_sha256": source_hash,
        "split_counts": counts,
        "selection_only": True,
        "holdout_evaluated": False,
    }


def load_corpus(path: Path) -> list[dict]:
    paths = sorted(
        item for item in path.rglob("*")
        if item.is_file() and item.suffix.lower() == ".json"
    )
    if not paths:
        raise ValueError(f"{path}: corpus directory contains no JSON files")
    documents = []
    for json_path in paths:
        value = read_json(json_path)
        values = value if isinstance(value, list) else [value]
        if not all(isinstance(document, dict) for document in values):
            raise ValueError(f"{json_path}: expected document object(s)")
        documents.extend(values)
    document_ids = [str(document.get("id")) for document in documents]
    if len(document_ids) != len(set(document_ids)):
        raise ValueError("corpus contains duplicate document IDs")
    return documents


def chunk_corpus(documents: list[dict]) -> list[dict]:
    if CHUNK_SIZE - CHUNK_OVERLAP != CHUNK_STEP or CHUNK_STEP <= 0:
        raise RuntimeError("fixed-window controls changed")
    chunks = []
    for document in documents:
        document_id = str(document["id"])
        passage = document.get("passage")
        if not isinstance(passage, str):
            raise TypeError(f"document {document_id!r}: passage must be a string")
        for chunk_index, start in enumerate(range(0, len(passage), CHUNK_STEP)):
            end = min(start + CHUNK_SIZE, len(passage))
            text = passage[start:end]
            chunks.append({
                "chunk_id": f"{document_id}:{chunk_index}",
                "document_id": document_id,
                "text": text,
                "char_start": start,
                "char_end": end,
            })
            if chunks[-1]["text"] != passage[start:end]:
                raise RuntimeError("source-preserving chunk invariant failed")
            if end == len(passage):
                break
    return chunks


def model_metadata(model, model_name: str, declared_revision: str, path: Path) -> dict:
    value = getattr(model.config, "_commit_hash", None)
    config_hash = value.strip() if isinstance(value, str) and value.strip() else None
    if config_hash is None:
        status = "declared-offline-snapshot"
    elif config_hash == declared_revision:
        status = "verified-from-config"
    else:
        raise RuntimeError(
            f"{model_name} config _commit_hash {config_hash!r} does not match "
            f"declared revision {declared_revision!r}"
        )
    return {
        "model_name": model_name,
        "declared_revision": declared_revision,
        "config_commit_hash": config_hash,
        "revision_status": status,
        "local_input_path": str(path),
    }


def load_dense_model() -> dict:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle CUDA accelerator")
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(DENSE_MODEL_PATH, local_files_only=True)
    model = AutoModel.from_pretrained(
        DENSE_MODEL_PATH, dtype=torch.float16, local_files_only=True
    )
    tokenizer_limit = int(tokenizer.model_max_length)
    model_limit = int(getattr(model.config, "max_position_embeddings", tokenizer_limit))
    if tokenizer_limit < DENSE_MAX_LENGTH or model_limit < DENSE_MAX_LENGTH:
        raise RuntimeError("local dense model does not support max_length=8192")
    metadata = model_metadata(
        model, DENSE_MODEL_NAME, DENSE_DECLARED_REVISION, DENSE_MODEL_PATH
    )
    model.to("cuda")
    model.eval()
    return {
        "tokenizer": tokenizer,
        "model": model,
        "metadata": metadata,
        "load_seconds": perf_counter() - started,
    }


def load_reranker() -> dict:
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(
        RERANKER_MODEL_PATH, local_files_only=True
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        RERANKER_MODEL_PATH, dtype=torch.float16, local_files_only=True
    )
    tokenizer_limit = int(tokenizer.model_max_length)
    model_limit = int(getattr(model.config, "max_position_embeddings", tokenizer_limit))
    if tokenizer_limit < RERANKER_MAX_SEQUENCE_LENGTH or model_limit < RERANKER_MAX_SEQUENCE_LENGTH:
        raise RuntimeError("local reranker does not support max_sequence_length=8192")
    metadata = model_metadata(
        model, RERANKER_MODEL_NAME, RERANKER_DECLARED_REVISION, RERANKER_MODEL_PATH
    )
    model.to("cuda")
    model.eval()
    return {
        "tokenizer": tokenizer,
        "model": model,
        "metadata": metadata,
        "load_seconds": perf_counter() - started,
    }


def encode_normalized_cls(model_bundle: dict, texts: list[str], batch_size: int) -> dict:
    embeddings = []
    started = perf_counter()
    for batch_start in range(0, len(texts), batch_size):
        batch = texts[batch_start:batch_start + batch_size]
        inputs = model_bundle["tokenizer"](
            batch,
            padding=True,
            truncation=True,
            max_length=DENSE_MAX_LENGTH,
            return_tensors="pt",
        )
        inputs = {name: value.to("cuda") for name, value in inputs.items()}
        with torch.no_grad():
            outputs = model_bundle["model"](**inputs, return_dict=True)
            embedding = F.normalize(outputs.last_hidden_state[:, 0], p=2, dim=1)
        if embedding.ndim != 2 or not torch.isfinite(embedding).all():
            raise RuntimeError("dense encoder returned invalid normalized CLS embeddings")
        embeddings.append(embedding.cpu())
    encoded = torch.cat(embeddings, dim=0)
    if encoded.shape[0] != len(texts):
        raise RuntimeError("dense embedding count mismatch")
    return {"embeddings": encoded, "seconds": perf_counter() - started}


def aggregate_dense_hits(raw_hits: list[tuple[float, int]], chunks: list[dict]) -> list[dict]:
    grouped = defaultdict(list)
    for chunk_rank, (score_value, chunk_index) in enumerate(raw_hits, start=1):
        score = float(score_value)
        if not isfinite(score):
            raise RuntimeError("dense retrieval produced a non-finite score")
        chunk = chunks[int(chunk_index)]
        grouped[chunk["document_id"]].append({
            "chunk_index": int(chunk_index),
            "dense_score": score,
            "chunk_rank": chunk_rank,
        })

    documents = []
    for document_id, hits in grouped.items():
        ordered = sorted(
            hits,
            key=lambda hit: (-hit["dense_score"], hit["chunk_rank"], hit["chunk_index"]),
        )
        documents.append({
            "document_id": document_id,
            "dense_document_score": sum(hit["dense_score"] for hit in ordered[:2]),
            "best_chunk_rank": ordered[0]["chunk_rank"],
            "available_supporting_chunks": len(ordered),
            "supporting_chunk_indices": [
                hit["chunk_index"] for hit in ordered[:max(SUPPORT_POOL_SIZES)]
            ],
            "supporting_dense_scores": [
                hit["dense_score"] for hit in ordered[:max(SUPPORT_POOL_SIZES)]
            ],
        })
    documents.sort(
        key=lambda document: (
            -document["dense_document_score"],
            document["best_chunk_rank"],
            document["document_id"],
        )
    )
    selected = documents[:CANDIDATE_DEPTH]
    if len(selected) != CANDIDATE_DEPTH:
        raise RuntimeError(f"expected {CANDIDATE_DEPTH} candidate documents")
    for original_rank, document in enumerate(selected, start=1):
        document["original_dense_rank"] = original_rank
        expected = min(max(SUPPORT_POOL_SIZES), document["available_supporting_chunks"])
        if len(document["supporting_chunk_indices"]) != expected:
            raise RuntimeError("support-pool construction mismatch")
    ids = [document["document_id"] for document in selected]
    if len(ids) != len(set(ids)):
        raise RuntimeError("candidate ranking contains duplicate document IDs")
    return selected


def retrieve_dense(
    query_embeddings: torch.Tensor,
    corpus_embeddings: torch.Tensor,
    chunks: list[dict],
    sample_ids: list[str],
) -> dict:
    if query_embeddings.shape[0] != len(sample_ids):
        raise ValueError("query embedding count mismatch")
    started = perf_counter()
    corpus_on_gpu = corpus_embeddings.to("cuda")
    candidates = {}
    for batch_start in range(0, len(sample_ids), QUERY_BATCH_SIZE):
        batch_ids = sample_ids[batch_start:batch_start + QUERY_BATCH_SIZE]
        query_on_gpu = query_embeddings[
            batch_start:batch_start + len(batch_ids)
        ].to("cuda")
        similarities = query_on_gpu @ corpus_on_gpu.T
        if not torch.isfinite(similarities).all():
            raise RuntimeError("dense similarity produced non-finite scores")
        top_scores, top_indices = torch.topk(
            similarities, k=TOP_K_CHUNKS, dim=1, largest=True, sorted=True
        )
        for row, sample_id in enumerate(batch_ids):
            raw_hits = list(zip(
                top_scores[row].float().cpu().tolist(),
                top_indices[row].cpu().tolist(),
            ))
            raw_hits.sort(key=lambda item: (-item[0], item[1]))
            candidates[sample_id] = aggregate_dense_hits(raw_hits, chunks)
    torch.cuda.synchronize()
    seconds = perf_counter() - started
    del corpus_on_gpu
    torch.cuda.empty_cache()
    rankings = {
        sample_id: [document["document_id"] for document in documents]
        for sample_id, documents in candidates.items()
    }
    return {"candidates": candidates, "rankings": rankings, "seconds": seconds}


def score_pair_records(reranker: dict, records: list[tuple[tuple, str, str]]) -> dict:
    scores = {}
    forward_seconds = 0.0
    total_started = perf_counter()
    for batch_start in range(0, len(records), RERANKER_BATCH_SIZE):
        batch = records[batch_start:batch_start + RERANKER_BATCH_SIZE]
        inputs = reranker["tokenizer"](
            [record[1] for record in batch],
            [record[2] for record in batch],
            padding=True,
            truncation="only_second",
            max_length=RERANKER_MAX_SEQUENCE_LENGTH,
            return_tensors="pt",
        )
        inputs = {name: value.to("cuda") for name, value in inputs.items()}
        torch.cuda.synchronize()
        forward_started = perf_counter()
        with torch.no_grad():
            logits = reranker["model"](**inputs, return_dict=True).logits.view(-1).float()
        torch.cuda.synchronize()
        forward_seconds += perf_counter() - forward_started
        values = logits.cpu().tolist()
        if len(values) != len(batch) or not all(isfinite(value) for value in values):
            raise RuntimeError("reranker returned invalid scores")
        for record, value in zip(batch, values):
            key = record[0]
            if key in scores:
                raise RuntimeError("duplicate CE pair key")
            scores[key] = float(value)
    return {
        "scores": scores,
        "pairs": len(records),
        "forward_seconds": forward_seconds,
        "total_seconds": perf_counter() - total_started,
    }


def build_pair_bands(samples: dict, candidates: dict, chunks: list[dict]) -> dict:
    bands = {"positions_1_2": [], "positions_3_4": [], "positions_5_8": []}
    bounds = {
        "positions_1_2": (0, 2),
        "positions_3_4": (2, 4),
        "positions_5_8": (4, 8),
    }
    for sample_id, sample in samples.items():
        for document in candidates[sample_id]:
            indices = document["supporting_chunk_indices"]
            for band_name, (start, stop) in bounds.items():
                for dense_position in range(start, min(stop, len(indices))):
                    chunk_index = indices[dense_position]
                    key = (sample_id, document["document_id"], chunk_index)
                    bands[band_name].append(
                        (key, sample["question"], chunks[chunk_index]["text"])
                    )
    return bands


def score_support_union(reranker: dict, pair_bands: dict) -> dict:
    cache = {}
    diagnostics = {}
    for band_name in ("positions_1_2", "positions_3_4", "positions_5_8"):
        scored = score_pair_records(reranker, pair_bands[band_name])
        if set(cache).intersection(scored["scores"]):
            raise RuntimeError("CE pair bands overlap")
        cache.update(scored["scores"])
        diagnostics[band_name] = {
            "pairs": scored["pairs"],
            "forward_seconds": scored["forward_seconds"],
            "total_seconds": scored["total_seconds"],
        }
    return {"cache": cache, "bands": diagnostics}


def derive_variant(samples: dict, candidates: dict, score_cache: dict, m: int) -> dict:
    rankings = {}
    selected_evidence = {}
    for sample_id in samples:
        ranked_documents = []
        selected_evidence[sample_id] = {}
        for document in candidates[sample_id]:
            support = document["supporting_chunk_indices"][:m]
            if not support:
                raise RuntimeError("candidate document has no supporting chunk")
            scored_chunks = []
            for dense_position, chunk_index in enumerate(support):
                key = (sample_id, document["document_id"], chunk_index)
                if key not in score_cache:
                    raise RuntimeError("missing CE score; no imputation is allowed")
                scored_chunks.append({
                    "chunk_index": chunk_index,
                    "dense_position": dense_position,
                    "ce_score": score_cache[key],
                })
            scored_chunks.sort(
                key=lambda item: (
                    -item["ce_score"], item["dense_position"], item["chunk_index"]
                )
            )
            chosen = scored_chunks[:CE_SELECTED_CHUNKS]
            selected_evidence[sample_id][document["document_id"]] = [
                item["chunk_index"] for item in chosen
            ]
            ranked_documents.append({
                "document_id": document["document_id"],
                "document_ce_score": sum(item["ce_score"] for item in chosen),
                "original_dense_rank": document["original_dense_rank"],
            })
        ranked_documents.sort(
            key=lambda item: (
                -item["document_ce_score"],
                item["original_dense_rank"],
                item["document_id"],
            )
        )
        ranking = [item["document_id"] for item in ranked_documents]
        if len(ranking) != CANDIDATE_DEPTH or len(ranking) != len(set(ranking)):
            raise RuntimeError("variant ranking must contain 100 unique candidate documents")
        rankings[sample_id] = ranking
    return {"rankings": rankings, "selected_evidence": selected_evidence}


def candidate_recall_at_100(samples: dict, rankings: dict) -> float:
    values = []
    for sample_id, sample in samples.items():
        gold = {str(document_id) for document_id in sample["answer"]}
        values.append(len(gold.intersection(rankings[sample_id][:100])) / len(gold))
    return float(np.mean(values))


def make_predictions(rankings: dict) -> dict:
    predictions = {}
    for sample_id, ranked in rankings.items():
        top_ids = [str(document_id) for document_id in ranked[:FINAL_K]]
        if len(top_ids) != FINAL_K or len(top_ids) != len(set(top_ids)):
            raise RuntimeError("top-5 prediction must contain five unique IDs; no deduplication")
        predictions[sample_id] = {"answer": top_ids}
    return predictions


def bundled_scorer_compatible_eval(predictions: dict, truth: dict) -> dict:
    y_pred = {key: value["answer"] for key, value in predictions.items()}
    y_true = {key: value for key, value in truth.items()}
    if set(y_pred) != set(y_true):
        raise RuntimeError("Samples in predictions do not match the reference")
    recall = np.array([
        len(set(y_true[key]) & set(y_pred[key])) / len(y_true[key])
        if 0 < len(y_pred[key]) <= 5 else 0 for key in y_true
    ]).mean()
    precision = np.array([
        len(set(y_true[key]) & set(y_pred[key])) / len(y_pred[key])
        if 0 < len(y_pred[key]) <= 5 else 0 for key in y_pred
    ]).mean()
    return {"precision": float(precision), "recall": float(recall)}


def internal_metrics(samples: dict, rankings: dict) -> dict:
    recalls = {depth: [] for depth in (5, 10, 20, 50, 100)}
    reciprocal_ranks = []
    for sample_id, sample in samples.items():
        ranked = rankings[sample_id]
        if len(ranked) != CANDIDATE_DEPTH or len(ranked) != len(set(ranked)):
            raise RuntimeError("final ranking must contain 100 unique IDs")
        gold = {str(document_id) for document_id in sample["answer"]}
        for depth, values in recalls.items():
            values.append(len(gold.intersection(ranked[:depth])) / len(gold))
        first = next(
            (rank for rank, doc_id in enumerate(ranked, 1) if doc_id in gold), None
        )
        reciprocal_ranks.append(0.0 if first is None else 1.0 / first)
    return {
        **{
            f"recall_at_{depth}": float(np.mean(values))
            for depth, values in recalls.items()
        },
        "mrr": float(np.mean(reciprocal_ranks)),
        "mrr_scope": "fixed top-100; absent gold gives reciprocal rank 0",
    }


def percentile(values: list[int], q: int) -> float | None:
    return None if not values else float(np.percentile(np.asarray(values), q))


def first_gold_summary(samples: dict, rankings: dict) -> dict:
    found = []
    bins = Counter()
    for sample_id, sample in samples.items():
        gold = {str(document_id) for document_id in sample["answer"]}
        first = next(
            (
                rank
                for rank, doc_id in enumerate(rankings[sample_id], 1)
                if doc_id in gold
            ),
            None,
        )
        if first is None:
            bins["not_found"] += 1
        else:
            found.append(first)
            label = (
                "rank_1" if first == 1 else "rank_2_5" if first <= 5
                else "rank_6_10" if first <= 10
                else "rank_11_20" if first <= 20
                else "rank_21_50" if first <= 50
                else "rank_51_100"
            )
            bins[label] += 1
    labels = (
        "rank_1", "rank_2_5", "rank_6_10", "rank_11_20",
        "rank_21_50", "rank_51_100", "not_found",
    )
    return {
        "when_found": {
            "median": float(median(found)) if found else None,
            "p90": percentile(found, 90),
            "p95": percentile(found, 95),
        },
        "counts": {label: bins[label] for label in labels},
    }


def per_query_top_5(samples: dict, rankings: dict) -> dict:
    precision = []
    recall = []
    for sample_id, sample in samples.items():
        gold = {str(document_id) for document_id in sample["answer"]}
        predicted = rankings[sample_id][:FINAL_K]
        overlap = len(gold.intersection(predicted))
        precision.append(overlap / len(predicted))
        recall.append(overlap / len(gold))
    return {"precision": np.asarray(precision), "recall": np.asarray(recall)}


def paired_behavior(control: dict, alternative: dict) -> dict:
    output = {}
    for metric in ("precision", "recall"):
        delta = alternative[metric] - control[metric]
        output[metric] = {
            "improved": int(np.sum(delta > 0)),
            "unchanged": int(np.sum(delta == 0)),
            "worsened": int(np.sum(delta < 0)),
        }
    return output


def paired_bootstrap(control: dict, alternative: dict) -> dict:
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    query_count = len(control["precision"])
    output = {}
    for metric in ("precision", "recall"):
        paired_delta = alternative[metric] - control[metric]
        samples = np.empty(BOOTSTRAP_RESAMPLES, dtype=np.float64)
        for start in range(0, BOOTSTRAP_RESAMPLES, 512):
            stop = min(start + 512, BOOTSTRAP_RESAMPLES)
            indices = rng.integers(
                0, query_count, size=(stop - start, query_count)
            )
            samples[start:stop] = paired_delta[indices].mean(axis=1)
        output[metric] = {
            "observed_delta": float(paired_delta.mean()),
            "bootstrap_mean": float(samples.mean()),
            "percentile_interval_95": [
                float(np.percentile(samples, 2.5)),
                float(np.percentile(samples, 97.5)),
            ],
            "fraction_delta_gt_0": float(np.mean(samples > 0)),
        }
    return {
        "ran": True,
        "seed": BOOTSTRAP_SEED,
        "resamples": BOOTSTRAP_RESAMPLES,
        **output,
    }


def support_pool_diagnostics(candidates: dict, query_count: int) -> dict:
    available_counts = [
        document["available_supporting_chunks"]
        for documents in candidates.values()
        for document in documents
    ]
    candidate_documents = len(available_counts)
    if candidate_documents != query_count * CANDIDATE_DEPTH:
        raise RuntimeError("candidate-document diagnostic count mismatch")
    bins = Counter()
    for count in available_counts:
        label = (
            "1" if count == 1 else "2" if count == 2 else "3" if count == 3
            else "4_7" if count <= 7 else "8_plus"
        )
        bins[label] += 1
    by_variant = {}
    for m in SUPPORT_POOL_SIZES:
        pairs = sum(min(m, count) for count in available_counts)
        by_variant[f"m{m}"] = {
            "candidate_documents": candidate_documents,
            "available_supporting_chunks": pairs,
            "total_ce_pairs": pairs,
            "mean_ce_pairs_per_query": pairs / query_count,
            "mean_chunks_per_document": pairs / candidate_documents,
        }
    return {
        "within_original_top_2000_chunk_pool": {
            "candidate_documents": candidate_documents,
            "one_available_dense_chunk": bins["1"],
            "two_available_dense_chunks": bins["2"],
            "three_available_dense_chunks": bins["3"],
            "four_to_seven_available_dense_chunks": bins["4_7"],
            "eight_or_more_available_dense_chunks": bins["8_plus"],
        },
        "by_variant": by_variant,
    }


def evidence_replacement_summary(candidates: dict, selected_evidence: dict) -> dict:
    counts = Counter()
    total = 0
    for sample_id, documents in candidates.items():
        for document in documents:
            dense_top_2 = set(document["supporting_chunk_indices"][:2])
            ce_top_2 = set(selected_evidence[sample_id][document["document_id"]])
            replacements = len(dense_top_2 - ce_top_2)
            if replacements not in (0, 1, 2):
                raise RuntimeError("invalid evidence-replacement count")
            counts[replacements] += 1
            total += 1
    return {
        "candidate_documents": total,
        "fraction_ce_top_2_differs_from_dense_top_2": (
            counts[1] + counts[2]
        ) / total,
        "replacement_counts": {
            "0_replacements": counts[0],
            "1_replacement": counts[1],
            "2_replacements": counts[2],
        },
    }


def metric_deltas(
    control_bundled: dict,
    control_internal: dict,
    alternative_bundled: dict,
    alternative_internal: dict,
) -> dict:
    return {
        "precision": alternative_bundled["precision"] - control_bundled["precision"],
        "recall": alternative_bundled["recall"] - control_bundled["recall"],
        "mrr": alternative_internal["mrr"] - control_internal["mrr"],
        "recall_at_10": (
            alternative_internal["recall_at_10"] - control_internal["recall_at_10"]
        ),
        "recall_at_20": (
            alternative_internal["recall_at_20"] - control_internal["recall_at_20"]
        ),
        "recall_at_50": (
            alternative_internal["recall_at_50"] - control_internal["recall_at_50"]
        ),
        "recall_at_100": (
            alternative_internal["recall_at_100"]
            - control_internal["recall_at_100"]
        ),
    }


def close_to(value: float, expected: float) -> bool:
    return abs(value - expected) <= SANITY_ABS_TOLERANCE


def conclusion_for(deltas: dict) -> str:
    improving = [
        name
        for name in ("m4_minus_m2", "m8_minus_m2")
        if deltas[name]["precision"] > SANITY_ABS_TOLERANCE
        and deltas[name]["recall"] > SANITY_ABS_TOLERANCE
    ]
    if len(improving) == 1:
        selected = improving[0].split("_")[0]
        return (
            f"{selected} improves both DEV precision and recall and is selected for "
            "later fixed-local-holdout validation; m2 remains the current validated "
            "reference."
        )
    if len(improving) == 2:
        m4 = deltas["m4_minus_m2"]
        m8 = deltas["m8_minus_m2"]
        if (
            m4["precision"] >= m8["precision"]
            and m4["recall"] >= m8["recall"]
            and (
                m4["precision"] > m8["precision"]
                or m4["recall"] > m8["recall"]
            )
        ):
            return (
                "m4 Pareto-dominates m8 on DEV and is selected for later "
                "fixed-local-holdout validation; m2 remains the current validated reference."
            )
        if (
            m8["precision"] >= m4["precision"]
            and m8["recall"] >= m4["recall"]
            and (
                m8["precision"] > m4["precision"]
                or m8["recall"] > m4["recall"]
            )
        ):
            return (
                "m8 Pareto-dominates m4 on DEV and is selected for later "
                "fixed-local-holdout validation; m2 remains the current validated reference."
            )
        return (
            "m4 and m8 both improve over m2 but trade precision against recall; "
            "record the trade-off and do not choose automatically without a competition objective."
        )
    tradeoff = any(
        deltas[name]["precision"] * deltas[name]["recall"] < 0
        for name in ("m4_minus_m2", "m8_minus_m2")
    )
    if tradeoff:
        return (
            "The larger support pools introduce a DEV precision/recall trade-off; "
            "do not choose automatically without a competition objective."
        )
    return (
        "Neither larger support pool improves both DEV precision and recall; keep the "
        "current top-2 dense supporting-chunk policy. Increasing the CE evidence pool "
        "alone does not explain the remaining ranking gap."
    )


In [ ]:
# Execute this cell manually on Kaggle; this repository task does not run it.
run_started = perf_counter()
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle CUDA accelerator")
torch.cuda.reset_peak_memory_stats()

controls = (
    CHUNK_SIZE,
    CHUNK_OVERLAP,
    CHUNK_STEP,
    TOP_K_CHUNKS,
    DOCUMENT_AGGREGATION,
    CANDIDATE_DEPTH,
    SUPPORT_POOL_SIZES,
    CE_SELECTED_CHUNKS,
    DENSE_MAX_LENGTH,
    RERANKER_MAX_SEQUENCE_LENGTH,
    RERANKER_BATCH_SIZE,
    FINAL_K,
)
expected_controls = (
    2_000,
    200,
    1_800,
    2_000,
    "sum_top_2_dense_chunk_scores",
    100,
    (2, 4, 8),
    2,
    8_192,
    8_192,
    128,
    5,
)
if controls != expected_controls:
    raise RuntimeError("a fixed control changed; stop")

# The loader returns only fixed DEV samples. No holdout evaluation or public inference exists here.
dev_samples, split_info = load_fixed_dev(LEGALIR_SOURCE_PATH)
documents = load_corpus(CORPUS_PATH)
chunks = chunk_corpus(documents)
if len(documents) != EXPECTED_DOCUMENTS or len(chunks) != EXPECTED_CHUNKS:
    raise RuntimeError(
        f"fixed corpus mismatch: got {len(documents)} documents / {len(chunks)} chunks"
    )

dense_model = load_dense_model()
dense_model_load_seconds = dense_model["load_seconds"]
dense_model_metadata = dict(dense_model["metadata"])
corpus_encoding = encode_normalized_cls(
    dense_model, [chunk["text"] for chunk in chunks], CORPUS_BATCH_SIZE
)
corpus_embeddings = corpus_encoding["embeddings"]
if not torch.allclose(
    torch.linalg.vector_norm(corpus_embeddings.float(), dim=1),
    torch.ones(len(corpus_embeddings)),
    atol=2e-3,
    rtol=0,
):
    raise RuntimeError("corpus embeddings are not L2-normalized")
query_encoding = encode_normalized_cls(
    dense_model,
    [sample["question"] for sample in dev_samples.values()],
    QUERY_BATCH_SIZE,
)
dense_query_encoding_seconds = query_encoding["seconds"]
dense_retrieval = retrieve_dense(
    query_encoding["embeddings"],
    corpus_embeddings,
    chunks,
    list(dev_samples),
)
candidate_recall = candidate_recall_at_100(dev_samples, dense_retrieval["rankings"])
if not close_to(candidate_recall, EXPECTED_DENSE_RECALL_AT_100):
    raise RuntimeError(f"dense candidate Recall@100 sanity mismatch: {candidate_recall}")

dense_model["model"].to("cpu")
del dense_model, corpus_embeddings, query_encoding
gc.collect()
torch.cuda.empty_cache()

support_availability = support_pool_diagnostics(
    dense_retrieval["candidates"], len(dev_samples)
)
reranker = load_reranker()
pair_bands = build_pair_bands(dev_samples, dense_retrieval["candidates"], chunks)
union_scoring = score_support_union(reranker, pair_bands)
score_cache = union_scoring["cache"]
expected_union_pairs = support_availability["by_variant"]["m8"]["total_ce_pairs"]
if len(score_cache) != expected_union_pairs:
    raise RuntimeError("union CE score-cache size mismatch")

# Validate the current m=2 control before deriving or comparing larger support pools.
truth = {sample_id: sample["answer"] for sample_id, sample in dev_samples.items()}
m2 = derive_variant(dev_samples, dense_retrieval["candidates"], score_cache, 2)
m2_bundled = bundled_scorer_compatible_eval(make_predictions(m2["rankings"]), truth)
m2_internal = internal_metrics(dev_samples, m2["rankings"])
if not close_to(m2_bundled["precision"], EXPECTED_M2_PRECISION):
    raise RuntimeError(f"m=2 precision sanity mismatch: {m2_bundled['precision']}")
if not close_to(m2_bundled["recall"], EXPECTED_M2_RECALL):
    raise RuntimeError(f"m=2 recall sanity mismatch: {m2_bundled['recall']}")
if not close_to(m2_internal["mrr"], EXPECTED_M2_MRR):
    raise RuntimeError(f"m=2 MRR sanity mismatch: {m2_internal['mrr']}")
if not close_to(m2_internal["recall_at_100"], EXPECTED_DENSE_RECALL_AT_100):
    raise RuntimeError("m=2 Recall@100 sanity mismatch")

# Only after the control passes are m=4 and m=8 derived from the same score cache.
variants = {"m2": m2}
for m in (4, 8):
    variants[f"m{m}"] = derive_variant(
        dev_samples, dense_retrieval["candidates"], score_cache, m
    )

candidate_sets = {
    name: {
        sample_id: frozenset(ranking)
        for sample_id, ranking in variant["rankings"].items()
    }
    for name, variant in variants.items()
}
if not (candidate_sets["m2"] == candidate_sets["m4"] == candidate_sets["m8"]):
    raise RuntimeError(
        "candidate document sets differ across support-pool variants; implementation bug"
    )
for name in variants:
    if any(
        candidate_sets[name][sample_id]
        != frozenset(dense_retrieval["rankings"][sample_id])
        for sample_id in dev_samples
    ):
        raise RuntimeError(f"{name} changed the fixed dense top-100 candidate set")

bundled = {"m2": m2_bundled}
internal = {"m2": m2_internal}
first_gold = {
    "m2": first_gold_summary(dev_samples, variants["m2"]["rankings"])
}
contributions = {
    "m2": per_query_top_5(dev_samples, variants["m2"]["rankings"])
}
for name in ("m4", "m8"):
    bundled[name] = bundled_scorer_compatible_eval(
        make_predictions(variants[name]["rankings"]), truth
    )
    internal[name] = internal_metrics(dev_samples, variants[name]["rankings"])
    first_gold[name] = first_gold_summary(
        dev_samples, variants[name]["rankings"]
    )
    contributions[name] = per_query_top_5(
        dev_samples, variants[name]["rankings"]
    )

recall_at_100_values = [
    internal[name]["recall_at_100"] for name in ("m2", "m4", "m8")
]
if len(set(recall_at_100_values)) != 1:
    raise RuntimeError("Recall@100 differs across variants; candidate-set invariant failed")
if not all(
    close_to(value, EXPECTED_DENSE_RECALL_AT_100)
    for value in recall_at_100_values
):
    raise RuntimeError(
        f"Recall@100 expected {EXPECTED_DENSE_RECALL_AT_100}, got {recall_at_100_values}"
    )

deltas = {
    "m4_minus_m2": metric_deltas(
        bundled["m2"], internal["m2"], bundled["m4"], internal["m4"]
    ),
    "m8_minus_m2": metric_deltas(
        bundled["m2"], internal["m2"], bundled["m8"], internal["m8"]
    ),
}
if any(delta["recall_at_100"] != 0.0 for delta in deltas.values()):
    raise RuntimeError("Recall@100 delta must be exactly zero")

paired_behavior_result = {
    "m4_vs_m2": paired_behavior(contributions["m2"], contributions["m4"]),
    "m8_vs_m2": paired_behavior(contributions["m2"], contributions["m8"]),
}
paired_bootstrap_result = {}
for name in ("m4", "m8"):
    delta = deltas[f"{name}_minus_m2"]
    if (
        delta["precision"] > SANITY_ABS_TOLERANCE
        and delta["recall"] > SANITY_ABS_TOLERANCE
    ):
        paired_bootstrap_result[f"{name}_vs_m2"] = paired_bootstrap(
            contributions["m2"], contributions[name]
        )
    else:
        paired_bootstrap_result[f"{name}_vs_m2"] = {
            "ran": False,
            "seed": BOOTSTRAP_SEED,
            "resamples": BOOTSTRAP_RESAMPLES,
            "reason": "variant did not improve both observed precision and recall",
        }

evidence_replacement = {
    "m4": evidence_replacement_summary(
        dense_retrieval["candidates"], variants["m4"]["selected_evidence"]
    ),
    "m8": evidence_replacement_summary(
        dense_retrieval["candidates"], variants["m8"]["selected_evidence"]
    ),
}

band_runtime = union_scoring["bands"]
forward_1_2 = band_runtime["positions_1_2"]["forward_seconds"]
forward_3_4 = band_runtime["positions_3_4"]["forward_seconds"]
forward_5_8 = band_runtime["positions_5_8"]["forward_seconds"]
total_ce_seconds = sum(
    value["total_seconds"] for value in band_runtime.values()
)

result = {
    "split": split_info,
    "controls": {
        "documents": len(documents),
        "chunks": len(chunks),
        "chunk_size_characters": CHUNK_SIZE,
        "overlap_characters": CHUNK_OVERLAP,
        "step_characters": CHUNK_STEP,
        "source_preserving_fixed_windows": True,
        "article_aware_chunking": False,
        "title_enrichment": False,
        "dense_top_k_chunks": TOP_K_CHUNKS,
        "dense_document_aggregation": "sum of top two dense chunk scores",
        "candidate_documents": CANDIDATE_DEPTH,
        "support_pool_sizes": list(SUPPORT_POOL_SIZES),
        "support_source": (
            "prefixes per candidate document from the original dense top-2000 chunk pool"
        ),
        "cross_encoder_document_aggregation": (
            "sum of top two CE-scored chunks from each support prefix"
        ),
        "final_k": FINAL_K,
        "fusion": None,
    },
    "models": {
        "dense": {
            **dense_model_metadata,
            "representation": "outputs.last_hidden_state[:, 0], then L2 normalization",
            "similarity": "dot product",
            "query_instruction": None,
            "max_length": DENSE_MAX_LENGTH,
            "dynamic_padding": True,
            "dtype": "float16",
            "device": "cuda",
        },
        "reranker": {
            **reranker["metadata"],
            "pair": "(question, supporting_chunk)",
            "max_sequence_length": RERANKER_MAX_SEQUENCE_LENGTH,
            "dtype": "float16",
            "batch_size": RERANKER_BATCH_SIZE,
            "device": "cuda",
        },
    },
    "candidate_invariants": {
        "constructed_once_per_query": True,
        "same_top_100_document_ids_for_m2_m4_m8": True,
        "candidate_recall_at_100": candidate_recall,
        "recall_at_100_identical_across_variants": True,
        "expected_recall_at_100": EXPECTED_DENSE_RECALL_AT_100,
    },
    "support_pool_availability": support_availability,
    "m2": {
        "bundled_scorer": bundled["m2"],
        "internal": internal["m2"],
        "first_gold_rank": first_gold["m2"],
        "pair_count": support_availability["by_variant"]["m2"],
        "control_reproduced": True,
    },
    "m4": {
        "bundled_scorer": bundled["m4"],
        "internal": internal["m4"],
        "first_gold_rank": first_gold["m4"],
        "pair_count": support_availability["by_variant"]["m4"],
    },
    "m8": {
        "bundled_scorer": bundled["m8"],
        "internal": internal["m8"],
        "first_gold_rank": first_gold["m8"],
        "pair_count": support_availability["by_variant"]["m8"],
    },
    "deltas": deltas,
    "paired_behavior": paired_behavior_result,
    "paired_bootstrap": paired_bootstrap_result,
    "evidence_replacement": evidence_replacement,
    "runtime": {
        "dense_model_load_seconds": dense_model_load_seconds,
        "dense_corpus_encoding_seconds": corpus_encoding["seconds"],
        "dense_query_encoding_seconds": dense_query_encoding_seconds,
        "dense_retrieval_seconds": dense_retrieval["seconds"],
        "reranker_model_load_seconds": reranker["load_seconds"],
        "ce_scoring_strategy": (
            "score disjoint top-8 support-position bands once and cache all scores"
        ),
        "ce_pairs_by_variant": {
            name: support_availability["by_variant"][name]["total_ce_pairs"]
            for name in ("m2", "m4", "m8")
        },
        "ce_incremental_forward_seconds": {
            "positions_1_2": forward_1_2,
            "positions_3_4": forward_3_4,
            "positions_5_8": forward_5_8,
        },
        "ce_forward_seconds_by_variant": {
            "m2": forward_1_2,
            "m4": forward_1_2 + forward_3_4,
            "m8": forward_1_2 + forward_3_4 + forward_5_8,
        },
        "ce_total_seconds_scored_once": total_ce_seconds,
        "total_runtime_seconds": perf_counter() - run_started,
        "peak_gpu_memory_bytes": int(torch.cuda.max_memory_allocated()),
        "corpus_batch_size": CORPUS_BATCH_SIZE,
        "query_batch_size": QUERY_BATCH_SIZE,
        "reranker_batch_size": RERANKER_BATCH_SIZE,
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
    },
    "conclusion": conclusion_for(deltas),
}
RESULT_PATH.write_text(
    json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(result, ensure_ascii=False, indent=2))
print("Saved aggregate-only DEV result:", RESULT_PATH)
